##### FUNCTION TO LOAD DATA FROM RAW TO LANDING
 `load_domain("domain-name")`

In [0]:
%run ./operations

In [0]:
from pyspark.sql.functions import lit, current_timestamp, col, explode, date_format
from pyspark.sql import Row
import uuid
from datetime import datetime


# ─── CONFIG ──────────────────────────────────────────────────────────────
team_name        = "team_lemma"
source_base_path = "abfss://raw@schwabdldevsa.dfs.core.windows.net"
target_base_path = f"/Volumes/charles_schwab_retailbrokerage_dev_{team_name}/landing/PWG/"
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

In [0]:
spark.sql("USE CATALOG charles_schwab_retailbrokerage_dev_team_lemma;")
spark.sql("USE SCHEMA landing")

In [0]:
# ─── SCHEMA CONSTANTS ────────────────────────────────────────────────────
DAILYMARKET_B1_COLS  = ["DM_DATE", "DM_S_SYMB", "DM_CLOSE", "DM_HIGH", "DM_LOW", "DM_VOL"]
DAILYMARKET_B23_COLS = ["DM_ACTION", "DM_RECID"] + DAILYMARKET_B1_COLS

WATCHHISTORY_B1_COLS  = ["W_C_ID", "W_S_SYMB", "W_DTS", "W_ACTION"]
WATCHHISTORY_B23_COLS = ["CDC_FLAG", "CDC_DSN"] + WATCHHISTORY_B1_COLS

CASHTXN_B1_COLS  = ["CT_CA_ID", "CT_DTS", "CT_AMT", "CT_NAME"]
CASHTXN_B23_COLS = ["CDC_FLAG", "CDC_DSN"] + CASHTXN_B1_COLS

TRADE_B1_COLS = [
    "T_ID", "T_DTS", "T_ST_ID", "T_TT_ID", "T_IS_CASH",
    "T_S_SYMB", "T_QTY", "T_BID_PRICE", "T_CA_ID", "T_EXEC_NAME",
    "T_TRADE_PRICE", "T_CHRG", "T_COMM", "T_TAX"
]
TRADE_B23_COLS = ["CDC_FLAG", "CDC_DSN"] + TRADE_B1_COLS

HOLDING_B1_COLS  = ["HH_H_T_ID", "HH_T_ID", "HH_BEFORE_QTY", "HH_AFTER_QTY"]
HOLDING_B23_COLS = ["CDC_FLAG", "CDC_DSN"] + HOLDING_B1_COLS

# Customer.txt — 32 cols, CDC_DSN at position 3 (confirmed)
CUSTOMER_B23_COLS = [
    "CDC_FLAG", "C_ID", "CDC_DSN", "C_TAX_ID", "C_ST_ID",
    "C_L_NAME", "C_F_NAME", "C_M_NAME", "C_GNDR", "C_TIER", "C_DOB",
    "C_ADLINE1", "C_ADLINE2", "C_ZIPCODE", "C_CITY", "C_STATE_PROV", "C_CTRY",
    "C_CTRY_1", "C_AREA_1", "C_LOCAL_1", "C_EXT_1",
    "C_CTRY_2", "C_AREA_2", "C_LOCAL_2", "C_EXT_2",
    "C_CTRY_3", "C_AREA_3", "C_LOCAL_3", "C_EXT_3",
    "C_EMAIL_1", "C_EMAIL_2",
    "C_LCL_TX_ID", "C_NAT_TX_ID"
]

# CustomerMgmt.xml — 36 flattened fields (confirmed)
CUSTOMERMGMT_COLS = [
    "ActionType", "ActionTS",
    "C_ID", "C_TAX_ID", "C_GNDR", "C_TIER", "C_DOB",
    "C_L_NAME", "C_F_NAME", "C_M_NAME",
    "C_ADLINE1", "C_ADLINE2", "C_ZIPCODE", "C_CITY", "C_STATE_PROV", "C_CTRY",
    "C_PRIM_EMAIL", "C_ALT_EMAIL",
    "C_CTRY_CODE_1", "C_AREA_CODE_1", "C_LOCAL_1", "C_EXT_1",
    "C_CTRY_CODE_2", "C_AREA_CODE_2", "C_LOCAL_2", "C_EXT_2",
    "C_CTRY_CODE_3", "C_AREA_CODE_3", "C_LOCAL_3", "C_EXT_3",
    "C_LCL_TX_ID", "C_NAT_TX_ID",
    "CA_ID", "CA_TAX_ST", "CA_B_ID", "CA_NAME"
]

# Prospect.json — 22 flattened fields from nested schema
PROSPECT_SELECT = [
    ("agency_id",             "p.agency_id"),
    ("first_name",            "p.personal.name.first_name"),
    ("last_name",             "p.personal.name.last_name"),
    ("middle_initial",        "p.personal.name.middle_initial"),
    ("gender",                "p.personal.demographics.gender"),
    ("age",                   "p.personal.demographics.age"),
    ("marital_status",        "p.personal.demographics.marital_status"),
    ("address_line1",         "p.contact.address.line1"),
    ("address_line2",         "p.contact.address.line2"),
    ("city",                  "p.contact.address.city"),
    ("state",                 "p.contact.address.state"),
    ("postal_code",           "p.contact.address.postal_code"),
    ("country",               "p.contact.address.country"),
    ("phone_full_number",     "p.contact.phone.full_number"),
    ("annual_income",         "p.financial.income.annual_income"),
    ("net_worth",             "p.financial.wealth.net_worth"),
    ("credit_rating",         "p.financial.credit.credit_rating"),
    ("number_credit_cards",   "p.financial.credit.number_credit_cards"),
    ("own_or_rent",           "p.lifestyle.housing.own_or_rent"),
    ("number_children",       "p.lifestyle.family.number_children"),
    ("number_cars",           "p.lifestyle.assets.number_cars"),
    ("employer_name",         "p.employment.employer_name"),
]


# ─── DOMAIN MAP ──────────────────────────────────────────────────────────
DOMAIN_MAP = {
    "control": [
        {"file_name": "BatchDate.txt", "landing_table": "batchdate",
         "batches": ["1", "2", "3"], "reader": "batchdate_text",
         "columns": ["batchdate", "batchid"]},
    ],
    "cross": [
        {"file_name": "Date.txt", "landing_table": "date",
         "batches": ["1"], "reader": "csv_header", "columns": None},
        {"file_name": "Time.txt", "landing_table": "time",
         "batches": ["1"], "reader": "csv_header", "columns": None},
        {"file_name": "StatusType.txt", "landing_table": "statustype",
         "batches": ["1"], "reader": "csv_pipe",
         "columns": ["ST_ID", "ST_NAME"]},
        {"file_name": "TaxRate.txt", "landing_table": "taxrate",
         "batches": ["1"], "reader": "csv_pipe",
         "columns": ["TX_ID", "TX_NAME", "TX_RATE"]},
        {"file_name": "Industry.txt", "landing_table": "industry",
         "batches": ["1"], "reader": "csv_pipe",
         "columns": ["IN_ID", "IN_NAME", "IN_SC_ID"]},
        {"file_name": "TradeType.txt", "landing_table": "tradetype",
         "batches": ["1"], "reader": "csv_pipe",
         "columns": ["TT_ID", "TT_NAME", "TT_IS_SELL", "TT_IS_MRKT"]},
    ],
    "market": [
        {"file_name": "FINWIRE*", "landing_table": "finwire",
         "batches": ["1"], "reader": "text", "columns": None},
        {"file_name": "DailyMarket.txt", "landing_table": "dailymarket",
         "batches": ["1", "2", "3"], "reader": "csv_pipe",
         "columns_by_batch": {"1": DAILYMARKET_B1_COLS,
                              "2": DAILYMARKET_B23_COLS,
                              "3": DAILYMARKET_B23_COLS}},
    ],
    "hr_broker": [
        {"file_name": "HR.csv", "landing_table": "hr",
         "batches": ["1"], "reader": "csv_comma",
         "columns": ["EMPLOYEE_ID", "MANAGER_ID", "LAST_NAME", "FIRST_NAME",
                     "MIDDLE_INITIAL", "JOB_CODE", "BRANCH_ID", "OFFICE", "PHONE"]},
    ],
    "customer": [
        {"file_name": "CustomerMgmt.xml", "landing_table": "customermgmt",
         "batches": ["1"], "reader": "xml_customermgmt",
         "columns": CUSTOMERMGMT_COLS},
        {"file_name": "Customer.txt", "landing_table": "customer",
         "batches": ["2", "3"], "reader": "csv_pipe",
         "columns": CUSTOMER_B23_COLS},
        {"file_name": "prospect.json", "landing_table": "prospect",
         "batches": ["1", "2", "3"], "reader": "json_prospect",
         "columns": [c[0] for c in PROSPECT_SELECT]},
        {"file_name": "WatchHistory.txt", "landing_table": "watchhistory",
         "batches": ["1", "2", "3"], "reader": "csv_pipe",
         "columns_by_batch": {"1": WATCHHISTORY_B1_COLS,
                              "2": WATCHHISTORY_B23_COLS,
                              "3": WATCHHISTORY_B23_COLS}},
    ],
    "account": [
        {"file_name": "Account.txt", "landing_table": "account",
         "batches": ["2", "3"], "reader": "csv_pipe",
         "columns": ["CDC_FLAG", "CDC_DSN", "CA_ID", "CA_C_ID",
                     "CA_B_ID", "CA_NAME", "CA_TAX_ST", "CA_ST_ID"]},
        {"file_name": "CashTransaction.txt", "landing_table": "cashtransaction",
         "batches": ["1", "2", "3"], "reader": "csv_pipe",
         "columns_by_batch": {"1": CASHTXN_B1_COLS,
                              "2": CASHTXN_B23_COLS,
                              "3": CASHTXN_B23_COLS}},
    ],
    "trade": [
        {"file_name": "Trade.txt", "landing_table": "trade",
         "batches": ["1", "2", "3"], "reader": "csv_pipe",
         "columns_by_batch": {"1": TRADE_B1_COLS,
                              "2": TRADE_B23_COLS,
                              "3": TRADE_B23_COLS}},
        {"file_name": "TradeHistory.txt", "landing_table": "trade_history",
         "batches": ["1"], "reader": "csv_pipe",
         "columns": ["TH_T_ID", "TH_DTS", "TH_ST_ID"]},
        {"file_name": "HoldingHistory.txt", "landing_table": "holdinghistory",
         "batches": ["1", "2", "3"], "reader": "csv_pipe",
         "columns_by_batch": {"1": HOLDING_B1_COLS,
                              "2": HOLDING_B23_COLS,
                              "3": HOLDING_B23_COLS}},
    ],
}


In [0]:
# ─── READERS ─────────────────────────────────────────────────────────────
def read_source(spark, source_path, reader, batch_id=None):
    if reader == "batchdate_text":
        # single line text → split into batchdate + batchid
        raw = spark.read.text(source_path)
        return raw.withColumn("batchdate", col("value")) \
                  .withColumn("batchid", lit(batch_id)) \
                  .drop("value")

    if reader == "csv_header":
        return (spark.read.option("header", "true").option("sep", "|")
                .option("inferSchema", "false").csv(source_path))

    if reader == "csv_pipe":
        return (spark.read.option("header", "false").option("sep", "|")
                .option("inferSchema", "false").csv(source_path))

    if reader == "csv_comma":
        return (spark.read.option("header", "false").option("sep", ",")
                .option("inferSchema", "false").csv(source_path))

    if reader == "text":
        return spark.read.text(source_path)

    if reader == "xml_customermgmt":
        # Adjust rowTag to match your actual XML structure
        return (spark.read.format("xml")
                .option("rowTag", "TPCDI:Action").load(source_path))

    if reader == "json_prospect":
        return spark.read.option("multiLine", "true").json(source_path)

    raise ValueError(f"Unknown reader '{reader}'")


# ─── TRANSFORMERS ────────────────────────────────────────────────────────
def apply_columns(df, column_list):
    if not column_list:
        return df
    current = df.columns
    if len(current) < len(column_list):
        raise ValueError(
            f"File produced {len(current)} cols, schema expects {len(column_list)}"
        )
    return df.toDF(*(column_list + current[len(column_list):]))


def flatten_prospect(df):
    """Explode prospect_batch.prospects array and select 22 flat fields."""
    exploded = (df.select(explode(col("prospect_batch.prospects")).alias("p")))
    select_exprs = [col(path).alias(name) for name, path in PROSPECT_SELECT]
    return exploded.select(*select_exprs)


def flatten_customermgmt(df, target_cols):
    """
    Fixed XML Parsing Logic: Extracts fields properly from nested structs 
    instead of looking for top-level columns which results in nulls.
    """
    return df.select(
        col("_ActionType").cast("string").alias("ActionType"),
        col("_ActionTS").cast("string").alias("ActionTS"),
        col("Customer._C_ID").cast("string").alias("C_ID"),
        col("Customer._C_TAX_ID").cast("string").alias("C_TAX_ID"),
        col("Customer._C_GNDR").cast("string").alias("C_GNDR"),
        col("Customer._C_TIER").cast("string").alias("C_TIER"),
        col("Customer._C_DOB").cast("string").alias("C_DOB"),
        col("Customer.Name.C_L_NAME").cast("string").alias("C_L_NAME"),
        col("Customer.Name.C_F_NAME").cast("string").alias("C_F_NAME"),
        col("Customer.Name.C_M_NAME").cast("string").alias("C_M_NAME"),
        col("Customer.Address.C_ADLINE1").cast("string").alias("C_ADLINE1"),
        col("Customer.Address.C_ADLINE2").cast("string").alias("C_ADLINE2"),
        col("Customer.Address.C_ZIPCODE").cast("string").alias("C_ZIPCODE"),
        col("Customer.Address.C_CITY").cast("string").alias("C_CITY"),
        col("Customer.Address.C_STATE_PROV").cast("string").alias("C_STATE_PROV"),
        col("Customer.Address.C_CTRY").cast("string").alias("C_CTRY"),
        col("Customer.ContactInfo.C_PRIM_EMAIL").cast("string").alias("C_PRIM_EMAIL"),
        col("Customer.ContactInfo.C_ALT_EMAIL").cast("string").alias("C_ALT_EMAIL"),
        
        col("Customer.ContactInfo.C_PHONE_1.C_CTRY_CODE").cast("string").alias("C_CTRY_CODE_1"),
        col("Customer.ContactInfo.C_PHONE_1.C_AREA_CODE").cast("string").alias("C_AREA_CODE_1"),
        col("Customer.ContactInfo.C_PHONE_1.C_LOCAL").cast("string").alias("C_LOCAL_1"),
        col("Customer.ContactInfo.C_PHONE_1.C_EXT").cast("string").alias("C_EXT_1"),
        
        col("Customer.ContactInfo.C_PHONE_2.C_CTRY_CODE").cast("string").alias("C_CTRY_CODE_2"),
        col("Customer.ContactInfo.C_PHONE_2.C_AREA_CODE").cast("string").alias("C_AREA_CODE_2"),
        col("Customer.ContactInfo.C_PHONE_2.C_LOCAL").cast("string").alias("C_LOCAL_2"),
        col("Customer.ContactInfo.C_PHONE_2.C_EXT").cast("string").alias("C_EXT_2"),
        
        col("Customer.ContactInfo.C_PHONE_3.C_CTRY_CODE").cast("string").alias("C_CTRY_CODE_3"),
        col("Customer.ContactInfo.C_PHONE_3.C_AREA_CODE").cast("string").alias("C_AREA_CODE_3"),
        col("Customer.ContactInfo.C_PHONE_3.C_LOCAL").cast("string").alias("C_LOCAL_3"),
        col("Customer.ContactInfo.C_PHONE_3.C_EXT").cast("string").alias("C_EXT_3"),
        
        col("Customer.TaxInfo.C_LCL_TX_ID").cast("string").alias("C_LCL_TX_ID"),
        col("Customer.TaxInfo.C_NAT_TX_ID").cast("string").alias("C_NAT_TX_ID"),
        col("Customer.Account._CA_ID").cast("string").alias("CA_ID"),
        col("Customer.Account._CA_TAX_ST").cast("string").alias("CA_TAX_ST"),
        col("Customer.Account.CA_B_ID").cast("string").alias("CA_B_ID"),
        col("Customer.Account.CA_NAME").cast("string").alias("CA_NAME")
    )


def resolve_columns(entry, batch_id):
    if "columns_by_batch" in entry:
        return entry["columns_by_batch"].get(batch_id)
    return entry.get("columns")


# ─── MAIN FUNCTION ───────────────────────────────────────────────────────
def load_domain(domain_name: str):
    domain_name = domain_name.lower().strip()
    if domain_name not in DOMAIN_MAP:
        raise ValueError(
            f"Invalid domain '{domain_name}'. Valid: {list(DOMAIN_MAP.keys())}"
        )

    recon_results = []

    for entry in DOMAIN_MAP[domain_name]:
        file_name     = entry["file_name"]
        landing_table = entry["landing_table"]
        reader        = entry["reader"]

        if file_name.startswith("FINWIRE"):
            file_list = []
            for f in dbutils.fs.ls(f"{source_base_path}/Batch1/"):
                name = f.name.rstrip("/")
                if (name.startswith("FINWIRE")
                        and "_audit" not in name
                        and not name.endswith(".csv")):
                    file_list.append((name, "1", name))
        else:
            file_list = [(file_name, b, landing_table) for b in entry["batches"]]

        for fname, batch_id, table_name in file_list:
            batch_folder = f"Batch{batch_id}"
            source_path  = f"{source_base_path}/{batch_folder}/{fname}"
            target_path  = f"{target_base_path}{batch_folder}/{table_name}"
            column_list  = resolve_columns(entry, batch_id)

            try:
                df = read_source(spark, source_path, reader, batch_id=batch_id)

                # ── apply schema / flatten based on reader type ──
                if reader == "json_prospect":
                    df = flatten_prospect(df)
                elif reader == "xml_customermgmt":
                    df = flatten_customermgmt(df, column_list)
                elif reader in ("csv_pipe", "csv_comma"):
                    df = apply_columns(df, column_list)
                # csv_header, text, batchdate_text → already named, skip

                source_count = df.count()
            

                df = (df.withColumn("_source_name",  lit(table_name))
                        .withColumn("_source_file",  lit(fname))
                        .withColumn("_batch_id",     lit(batch_folder))
                        .withColumn("_ingestion_ts", current_timestamp())
                        .withColumn("_run_id", lit(run_id)))

                df.write.mode("overwrite").parquet(target_path)
                target_count = spark.read.parquet(target_path).count()
                status = "MATCH" if source_count == target_count else "MISMATCH"

                recon_results.append(Row(
                    source_table=table_name, batch_id=batch_folder,
                    target_path=target_path, source_count=source_count,
                    target_count=target_count, status=status,
                    columns_applied=",".join(column_list) if column_list else "AUTO"
                ))
                print(f"{table_name} ({batch_folder}) -> {status}  rows={source_count}")

            except Exception as e:
                recon_results.append(Row(
                    source_table=table_name, batch_id=batch_folder,
                    target_path=target_path, source_count=None,
                    target_count=None, status=f"ERROR: {str(e)[:200]}",
                    columns_applied=",".join(column_list) if column_list else "AUTO"
                ))
                print(f"Error {table_name} ({batch_folder}): {str(e)}")

    recon_df = spark.createDataFrame(recon_results)
    for row in recon_results:
        if row.status in ("MATCH", "MISMATCH"):
            # row.batch_id is "Batch1", etc. We pass it directly as STRING now!
            log_pipeline_recon(
                spark=spark,
                run_id=run_id,
                batch_id=row.batch_id,
                domain=domain_name.upper(),
                table_name=row.source_table,
                source_layer="raw",
                target_layer="landing",
                source_count=row.source_count,
                target_count=row.target_count
            )
            
            log_audit_event(
                spark=spark,
                run_id=run_id,
                batch=row.batch_id,
                layer="landing",
                table_name=row.source_table,
                operation="APPEND",
                rows_affected=row.target_count
            )
    display(recon_df)
    return recon_df



